# 13 - Build Canonical POIs

This notebook creates a canonical POI layer for the Istanbul tourism pilot.

Goals:
- start from enriched OSM POIs
- focus on important landmark-like POIs
- apply lightweight text normalization and grouping
- create canonical landmark/entity tables for later use in:
  - BestTime integration
  - Wikipedia enrichment
  - POI-level crowd computation
  - recommendation


In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load base files

In [2]:
pois = pd.read_csv("../data/processed/poi_enriched.csv")
grouping_candidates = pd.read_csv("../data/processed/poi_grouping_candidates.csv")

wiki_review_path = "../data/processed/wiki_review_candidates_top50.csv"
try:
    wiki_review = pd.read_csv(wiki_review_path)
except FileNotFoundError:
    wiki_review = pd.DataFrame()

print("pois:", pois.shape)
print("grouping_candidates:", grouping_candidates.shape)
print("wiki_review:", wiki_review.shape)

pois: (2761, 20)
grouping_candidates: (113, 19)
wiki_review: (25, 10)


## Keep only important landmark-like POIs
We focus on the high-value pilot subset first.

In [3]:
TARGET_CATEGORIES = ["museum", "historic", "attraction"]
TOP_N = 150

seed_pois = (
    pois[pois["category_clean"].isin(TARGET_CATEGORIES)]
    .sort_values("importance_score", ascending=False)
    .head(TOP_N)
    .copy()
)

print("Seed POIs:", seed_pois.shape)
seed_pois[["poi_id", "name", "name_en", "category_clean", "importance_score"]].head(20)

Seed POIs: (150, 20)


,poi_id,name,name_en,category_clean,importance_score
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,historic,0.863004
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,historic,0.859224
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,historic,0.858915
3,311681431,Sağlık Müzesi,NaN,museum,0.849310
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,museum,0.844213
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,attraction,0.835898
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,museum,0.834391
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,museum,0.834238
8,3373094254,Halı Müzesi,Carpet Museum,museum,0.827180
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,historic,0.821057


## NLP-style normalization helpers
This is the lightweight text-processing layer used for canonicalization.

In [4]:
GENERIC_WORDS = {
    "museum", "müzesi", "muzesi", "muze",
    "mosque", "cami", "camii",
    "palace", "saray", "sarayı", "sarayi",
    "tower", "kule",
    "square", "meydan", "meydani", "meydanı",
    "cistern", "sarnic", "sarnıcı", "sarnici",
    "church", "kilise",
    "tomb", "türbe", "turbe",
    "mausoleum",
    "history", "experience",
    "site", "park",
    "of", "the", "and"
}


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).casefold()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def meaningful_tokens(text):
    return [
        token
        for token in normalize_text(text).split()
        if token not in GENERIC_WORDS and len(token) >= 3
    ]


def root_name(text):
    return " ".join(meaningful_tokens(text))


In [5]:
seed_pois["canonical_seed_name"] = seed_pois["name_en"].fillna(seed_pois["name"])

seed_pois["name_norm"] = seed_pois["name"].apply(normalize_text)
seed_pois["name_en_norm"] = seed_pois["name_en"].fillna("").apply(normalize_text)
seed_pois["seed_name_norm"] = seed_pois["canonical_seed_name"].apply(normalize_text)

seed_pois["name_root"] = seed_pois["name"].apply(root_name)
seed_pois["name_en_root"] = seed_pois["name_en"].fillna("").apply(root_name)

seed_pois[[
    "poi_id",
    "name",
    "name_en",
    "canonical_seed_name",
    "name_root",
    "name_en_root"
]].head(20)

,poi_id,name,name_en,canonical_seed_name,name_root,name_en_root
0,13615790287,Lausos Sarayı'nın Kalıntıları,NaN,Lausos Sarayı'nın Kalıntıları,lausos kal lar,
1,1075801479,Antiochos Sarayı'nın Kalıntıları,NaN,Antiochos Sarayı'nın Kalıntıları,antiochos kal lar,
2,8120955,İbrahim Paşa Sarayı,Ibrahim Pasha Palace,Ibrahim Pasha Palace,ibrahim pasa,ibrahim pasha
3,311681431,Sağlık Müzesi,NaN,Sağlık Müzesi,sagl,
4,1153966162,Büyük Saray Mozaikleri Müzesi,Great Palace Mosaic Museum,Great Palace Mosaic Museum,buyuk mozaikleri,great mosaic
5,103953125,Sultan Ahmet Türbesi,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,sultan ahmet turbesi,sultan ahmed
6,11867279469,Ayasofya Tarih ve Deneyim Müzesi,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,ayasofya tarih deneyim,hagia sophia
7,5113500256,Türk ve İslam Eserleri Müzesi,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,turk islam eserleri,turkish islamic arts
8,3373094254,Halı Müzesi,Carpet Museum,Carpet Museum,hal,carpet
9,527580309,Keçecizâde Fuad Paşa Türbesi,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,kececizade fuad pasa turbesi,kececizade fuad pasha


## Seed root-name sanity check

In [6]:
seed_pois[["canonical_seed_name", "name_root", "name_en_root", "category_clean"]].head(30)

,canonical_seed_name,name_root,name_en_root,category_clean
0,Lausos Sarayı'nın Kalıntıları,lausos kal lar,,historic
1,Antiochos Sarayı'nın Kalıntıları,antiochos kal lar,,historic
2,Ibrahim Pasha Palace,ibrahim pasa,ibrahim pasha,historic
3,Sağlık Müzesi,sagl,,museum
4,Great Palace Mosaic Museum,buyuk mozaikleri,great mosaic,museum
5,Tomb of Sultan Ahmed I,sultan ahmet turbesi,sultan ahmed,attraction
6,Hagia Sophia History and Experience Museum,ayasofya tarih deneyim,hagia sophia,museum
7,Turkish and Islamic Arts Museum,turk islam eserleri,turkish islamic arts,museum
8,Carpet Museum,hal,carpet,museum
9,Turbe of Keçecizâde Fuad Pasha,kececizade fuad pasa turbesi,kececizade fuad pasha,historic


## Conservative grouping logic
We only merge very obvious duplicates or same-entity pairs.

In [7]:
class UnionFind:
    def __init__(self, values):
        self.parent = {v: v for v in values}

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]

        while self.parent[x] != x:
            nxt = self.parent[x]
            self.parent[x] = root
            x = nxt

        return root

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra != rb:
            self.parent[rb] = ra

In [8]:
def is_safe_same_entity(row, poi_index, poi_lookup):
    poi_1 = int(row["poi_id_1"])
    poi_2 = int(row["poi_id_2"])

    if poi_1 not in poi_index or poi_2 not in poi_index:
        return False

    if row["root_overlap_score"] < 1.0 or row["distance_km"] > 0.10:
        return False

    if row["same_category"] == 1:
        return True

    left = poi_lookup.loc[poi_1]
    right = poi_lookup.loc[poi_2]

    exact_seed_name = left["seed_name_norm"] and left["seed_name_norm"] == right["seed_name_norm"]
    exact_en_root = left["name_en_root"] and left["name_en_root"] == right["name_en_root"]
    exact_local_root = left["name_root"] and left["name_root"] == right["name_root"]

    return bool(exact_seed_name or exact_en_root or exact_local_root)

## Apply safe merges

In [9]:
uf = UnionFind(seed_pois["poi_id"].astype(int).tolist())

poi_index = set(seed_pois["poi_id"].astype(int))
poi_lookup = seed_pois.set_index("poi_id")

safe_merge_pairs = []

for _, row in grouping_candidates.iterrows():
    if is_safe_same_entity(row, poi_index, poi_lookup):
        uf.union(int(row["poi_id_1"]), int(row["poi_id_2"]))
        safe_merge_pairs.append(row)

safe_merge_pairs_df = pd.DataFrame(safe_merge_pairs)
print("Safe merge pairs:", safe_merge_pairs_df.shape)
safe_merge_pairs_df.head(20)

Safe merge pairs: (4, 19)


,poi_id_1,name_1,name_en_1,category_1,poi_id_2,name_2,name_en_2,category_2,cluster_id,distance_km,root_overlap_score,same_category,importance_1,importance_2,group_decision,group_type,family_name,entity_name,notes
1,247442912,Masumiyet Müzesi,NaN,museum,3279398393,Masumiyet Müzesi,The Museum of Innocence,museum,12,0.003800,1.0,1,0.694032,0.694025,NaN,NaN,NaN,NaN,NaN
4,499944905,Atik Ali Paşa Medresesi,NaN,historic,3374681017,Atik Ali Paşa Medresesi,NaN,historic,12,0.008571,1.0,1,0.694578,0.687220,NaN,NaN,NaN,NaN,NaN
5,1555271,Ayasofya-i Kebir Câmi-i Şerifi,NaN,museum,109862851,Ayasofya-i Kebir Câmi-i Şerifi,Hagia Sophia,attraction,12,0.012460,1.0,0,0.768184,0.747139,NaN,NaN,NaN,NaN,NaN
10,1184924695,Yerebatan Sarnıcı,NaN,historic,11006008339,Yerebatan Sarnıcı,Basilica Cistern (Exit),attraction,12,0.075361,1.0,0,0.785024,0.769588,NaN,NaN,NaN,NaN,NaN


In [10]:
seed_pois["group_root"] = seed_pois["poi_id"].astype(int).apply(uf.find)
seed_pois[["poi_id", "canonical_seed_name", "group_root"]].head(20)

,poi_id,canonical_seed_name,group_root
0,13615790287,Lausos Sarayı'nın Kalıntıları,13615790287
1,1075801479,Antiochos Sarayı'nın Kalıntıları,1075801479
2,8120955,Ibrahim Pasha Palace,8120955
3,311681431,Sağlık Müzesi,311681431
4,1153966162,Great Palace Mosaic Museum,1153966162
5,103953125,Tomb of Sultan Ahmed I,103953125
6,11867279469,Hagia Sophia History and Experience Museum,11867279469
7,5113500256,Turkish and Islamic Arts Museum,5113500256
8,3373094254,Carpet Museum,3373094254
9,527580309,Turbe of Keçecizâde Fuad Pasha,527580309


## Pick one representative per group
The most important POI in each group becomes the canonical representative.

In [11]:
representatives = {}

for group_root, group_df in seed_pois.groupby("group_root"):
    representatives[group_root] = group_df.sort_values(
        ["importance_score", "nearby_count_500m"],
        ascending=[False, False]
    ).iloc[0]


In [12]:
seed_pois["canonical_poi_id"] = seed_pois["group_root"].map(
    lambda root: f"cp_{int(representatives[root]['poi_id'])}"
)

seed_pois["canonical_name"] = seed_pois["group_root"].map(
    lambda root: representatives[root]["canonical_seed_name"]
)

seed_pois["family_name"] = seed_pois["canonical_name"]

seed_pois["group_method"] = seed_pois["group_root"].map(
    lambda root: "auto_grouped" if (seed_pois["group_root"] == root).sum() > 1 else "single_seed"
)

seed_pois["member_role"] = seed_pois.apply(
    lambda row: "primary"
    if row["poi_id"] == representatives[row["group_root"]]["poi_id"]
    else "member",
    axis=1
)

seed_pois[[
    "poi_id",
    "canonical_poi_id",
    "canonical_name",
    "family_name",
    "member_role",
    "group_method"
]].head(20)


,poi_id,canonical_poi_id,canonical_name,family_name,member_role,group_method
0,13615790287,cp_13615790287,Lausos Sarayı'nın Kalıntıları,Lausos Sarayı'nın Kalıntıları,primary,single_seed
1,1075801479,cp_1075801479,Antiochos Sarayı'nın Kalıntıları,Antiochos Sarayı'nın Kalıntıları,primary,single_seed
2,8120955,cp_8120955,Ibrahim Pasha Palace,Ibrahim Pasha Palace,primary,single_seed
3,311681431,cp_311681431,Sağlık Müzesi,Sağlık Müzesi,primary,single_seed
4,1153966162,cp_1153966162,Great Palace Mosaic Museum,Great Palace Mosaic Museum,primary,single_seed
5,103953125,cp_103953125,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,primary,single_seed
6,11867279469,cp_11867279469,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,primary,single_seed
7,5113500256,cp_5113500256,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,primary,single_seed
8,3373094254,cp_3373094254,Carpet Museum,Carpet Museum,primary,single_seed
9,527580309,cp_527580309,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,primary,single_seed


## Build canonical membership table

In [13]:
canonical_poi_members = seed_pois[[
    "poi_id",
    "canonical_poi_id",
    "canonical_name",
    "family_name",
    "category_clean",
    "cluster_id",
    "importance_score",
    "member_role",
    "group_method",
    "canonical_seed_name"
]].copy()

canonical_poi_members.head(20)

,poi_id,canonical_poi_id,canonical_name,family_name,category_clean,cluster_id,importance_score,member_role,group_method,canonical_seed_name
0,13615790287,cp_13615790287,Lausos Sarayı'nın Kalıntıları,Lausos Sarayı'nın Kalıntıları,historic,12,0.863004,primary,single_seed,Lausos Sarayı'nın Kalıntıları
1,1075801479,cp_1075801479,Antiochos Sarayı'nın Kalıntıları,Antiochos Sarayı'nın Kalıntıları,historic,12,0.859224,primary,single_seed,Antiochos Sarayı'nın Kalıntıları
2,8120955,cp_8120955,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,12,0.858915,primary,single_seed,Ibrahim Pasha Palace
3,311681431,cp_311681431,Sağlık Müzesi,Sağlık Müzesi,museum,12,0.849310,primary,single_seed,Sağlık Müzesi
4,1153966162,cp_1153966162,Great Palace Mosaic Museum,Great Palace Mosaic Museum,museum,12,0.844213,primary,single_seed,Great Palace Mosaic Museum
5,103953125,cp_103953125,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,attraction,12,0.835898,primary,single_seed,Tomb of Sultan Ahmed I
6,11867279469,cp_11867279469,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,museum,12,0.834391,primary,single_seed,Hagia Sophia History and Experience Museum
7,5113500256,cp_5113500256,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,museum,12,0.834238,primary,single_seed,Turkish and Islamic Arts Museum
8,3373094254,cp_3373094254,Carpet Museum,Carpet Museum,museum,12,0.827180,primary,single_seed,Carpet Museum
9,527580309,cp_527580309,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,historic,12,0.821057,primary,single_seed,Turbe of Keçecizâde Fuad Pasha


## Build canonical POI table

In [14]:
canonical_pois = (
    seed_pois.groupby("canonical_poi_id")
    .agg(
        canonical_name=("canonical_name", "first"),
        family_name=("family_name", "first"),
        category_clean=("category_clean", "first"),
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        cluster_id=("cluster_id", "first"),
        importance_score=("importance_score", "max"),
        member_count=("poi_id", "count"),
        member_poi_ids=("poi_id", lambda s: "|".join(str(int(v)) for v in sorted(s))),
        source_names=("canonical_seed_name", lambda s: " | ".join(dict.fromkeys(s.astype(str)))),
        group_method=("group_method", "first"),
    )
    .reset_index()
    .sort_values(["importance_score", "member_count"], ascending=[False, False])
    .reset_index(drop=True)
)

canonical_pois["wiki_title"] = np.nan
canonical_pois["wiki_match_type"] = np.nan
canonical_pois["wiki_match_quality"] = np.nan
canonical_pois["besttime_supported"] = np.nan
canonical_pois["notes"] = ""

canonical_pois.head(20)

,canonical_poi_id,canonical_name,family_name,category_clean,lat,lon,cluster_id,importance_score,member_count,member_poi_ids,source_names,group_method,wiki_title,wiki_match_type,wiki_match_quality,besttime_supported,notes
0,cp_13615790287,Lausos Sarayı'nın Kalıntıları,Lausos Sarayı'nın Kalıntıları,historic,41.007586,28.975554,12,0.863004,1,13615790287,Lausos Sarayı'nın Kalıntıları,single_seed,NaN,NaN,NaN,NaN,
1,cp_1075801479,Antiochos Sarayı'nın Kalıntıları,Antiochos Sarayı'nın Kalıntıları,historic,41.007289,28.975329,12,0.859224,1,1075801479,Antiochos Sarayı'nın Kalıntıları,single_seed,NaN,NaN,NaN,NaN,
2,cp_8120955,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,41.006397,28.974808,12,0.858915,1,8120955,Ibrahim Pasha Palace,single_seed,NaN,NaN,NaN,NaN,
3,cp_311681431,Sağlık Müzesi,Sağlık Müzesi,museum,41.008314,28.975290,12,0.849310,1,311681431,Sağlık Müzesi,single_seed,NaN,NaN,NaN,NaN,
4,cp_1153966162,Great Palace Mosaic Museum,Great Palace Mosaic Museum,museum,41.004295,28.977433,12,0.844213,1,1153966162,Great Palace Mosaic Museum,single_seed,NaN,NaN,NaN,NaN,
5,cp_103953125,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,attraction,41.006789,28.976985,12,0.835898,1,103953125,Tomb of Sultan Ahmed I,single_seed,NaN,NaN,NaN,NaN,
6,cp_11867279469,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,museum,41.006455,28.975367,12,0.834391,1,11867279469,Hagia Sophia History and Experience Museum,single_seed,NaN,NaN,NaN,NaN,
7,cp_5113500256,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,museum,41.006280,28.974915,12,0.834238,1,5113500256,Turkish and Islamic Arts Museum,single_seed,NaN,NaN,NaN,NaN,
8,cp_3373094254,Carpet Museum,Carpet Museum,museum,41.005682,28.978669,12,0.827180,1,3373094254,Carpet Museum,single_seed,NaN,NaN,NaN,NaN,
9,cp_527580309,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,historic,41.006561,28.972843,12,0.821057,1,527580309,Turbe of Keçecizâde Fuad Pasha,single_seed,NaN,NaN,NaN,NaN,


## Attach reviewed Wikipedia matches if available
Only reviewed `exact` / `parent` matches should be merged.

In [15]:
if not wiki_review.empty and "wiki_match_type" in wiki_review.columns:
    reviewed_wiki = wiki_review[wiki_review["wiki_match_type"].isin(["exact", "parent"])].copy()

    if not reviewed_wiki.empty:
        reviewed_wiki = reviewed_wiki.merge(
            canonical_poi_members[["poi_id", "canonical_poi_id"]],
            on="poi_id",
            how="inner"
        )

        if not reviewed_wiki.empty:
            reviewed_wiki = (
                reviewed_wiki.sort_values(
                    ["canonical_poi_id", "wiki_match_quality", "wiki_overlap_score"],
                    ascending=[True, True, False]
                )
                .groupby("canonical_poi_id", as_index=False)
                .first()
            )

            canonical_pois = canonical_pois.drop(
                columns=["wiki_title", "wiki_match_type", "wiki_match_quality"],
                errors="ignore"
            ).merge(
                reviewed_wiki[[
                    "canonical_poi_id",
                    "wiki_title",
                    "wiki_match_type",
                    "wiki_match_quality"
                ]],
                on="canonical_poi_id",
                how="left"
            )

canonical_pois.head(20)

,canonical_poi_id,canonical_name,family_name,category_clean,lat,lon,cluster_id,importance_score,member_count,member_poi_ids,source_names,group_method,wiki_title,wiki_match_type,wiki_match_quality,besttime_supported,notes
0,cp_13615790287,Lausos Sarayı'nın Kalıntıları,Lausos Sarayı'nın Kalıntıları,historic,41.007586,28.975554,12,0.863004,1,13615790287,Lausos Sarayı'nın Kalıntıları,single_seed,NaN,NaN,NaN,NaN,
1,cp_1075801479,Antiochos Sarayı'nın Kalıntıları,Antiochos Sarayı'nın Kalıntıları,historic,41.007289,28.975329,12,0.859224,1,1075801479,Antiochos Sarayı'nın Kalıntıları,single_seed,NaN,NaN,NaN,NaN,
2,cp_8120955,Ibrahim Pasha Palace,Ibrahim Pasha Palace,historic,41.006397,28.974808,12,0.858915,1,8120955,Ibrahim Pasha Palace,single_seed,NaN,NaN,NaN,NaN,
3,cp_311681431,Sağlık Müzesi,Sağlık Müzesi,museum,41.008314,28.975290,12,0.849310,1,311681431,Sağlık Müzesi,single_seed,NaN,NaN,NaN,NaN,
4,cp_1153966162,Great Palace Mosaic Museum,Great Palace Mosaic Museum,museum,41.004295,28.977433,12,0.844213,1,1153966162,Great Palace Mosaic Museum,single_seed,NaN,NaN,NaN,NaN,
5,cp_103953125,Tomb of Sultan Ahmed I,Tomb of Sultan Ahmed I,attraction,41.006789,28.976985,12,0.835898,1,103953125,Tomb of Sultan Ahmed I,single_seed,NaN,NaN,NaN,NaN,
6,cp_11867279469,Hagia Sophia History and Experience Museum,Hagia Sophia History and Experience Museum,museum,41.006455,28.975367,12,0.834391,1,11867279469,Hagia Sophia History and Experience Museum,single_seed,NaN,NaN,NaN,NaN,
7,cp_5113500256,Turkish and Islamic Arts Museum,Turkish and Islamic Arts Museum,museum,41.006280,28.974915,12,0.834238,1,5113500256,Turkish and Islamic Arts Museum,single_seed,NaN,NaN,NaN,NaN,
8,cp_3373094254,Carpet Museum,Carpet Museum,museum,41.005682,28.978669,12,0.827180,1,3373094254,Carpet Museum,single_seed,NaN,NaN,NaN,NaN,
9,cp_527580309,Turbe of Keçecizâde Fuad Pasha,Turbe of Keçecizâde Fuad Pasha,historic,41.006561,28.972843,12,0.821057,1,527580309,Turbe of Keçecizâde Fuad Pasha,single_seed,NaN,NaN,NaN,NaN,


## Inspect grouped canonical POIs

In [16]:
canonical_pois[canonical_pois["member_count"] > 1][[
    "canonical_poi_id",
    "canonical_name",
    "member_count",
    "member_poi_ids",
    "group_method"
]]

,canonical_poi_id,canonical_name,member_count,member_poi_ids,group_method
36,cp_1184924695,Yerebatan Sarnıcı,2,1184924695|11006008339,auto_grouped
46,cp_1555271,Ayasofya-i Kebir Câmi-i Şerifi,2,1555271|109862851,auto_grouped
106,cp_499944905,Atik Ali Paşa Medresesi,2,499944905|3374681017,auto_grouped
108,cp_247442912,Masumiyet Müzesi,2,247442912|3279398393,auto_grouped


In [17]:
grouped_ids = canonical_pois.loc[canonical_pois["member_count"] > 1, "canonical_poi_id"]

canonical_poi_members[
    canonical_poi_members["canonical_poi_id"].isin(grouped_ids)
].sort_values(
    ["canonical_poi_id", "member_role", "importance_score"],
    ascending=[True, True, False]
)

,poi_id,canonical_poi_id,canonical_name,family_name,category_clean,cluster_id,importance_score,member_role,group_method,canonical_seed_name
45,11006008339,cp_1184924695,Yerebatan Sarnıcı,Yerebatan Sarnıcı,attraction,12,0.769588,member,auto_grouped,Basilica Cistern (Exit)
36,1184924695,cp_1184924695,Yerebatan Sarnıcı,Yerebatan Sarnıcı,historic,12,0.785024,primary,auto_grouped,Yerebatan Sarnıcı
56,109862851,cp_1555271,Ayasofya-i Kebir Câmi-i Şerifi,Ayasofya-i Kebir Câmi-i Şerifi,attraction,12,0.747139,member,auto_grouped,Hagia Sophia
47,1555271,cp_1555271,Ayasofya-i Kebir Câmi-i Şerifi,Ayasofya-i Kebir Câmi-i Şerifi,museum,12,0.768184,primary,auto_grouped,Ayasofya-i Kebir Câmi-i Şerifi
111,3279398393,cp_247442912,Masumiyet Müzesi,Masumiyet Müzesi,museum,12,0.694025,member,auto_grouped,The Museum of Innocence
110,247442912,cp_247442912,Masumiyet Müzesi,Masumiyet Müzesi,museum,12,0.694032,primary,auto_grouped,Masumiyet Müzesi
120,3374681017,cp_499944905,Atik Ali Paşa Medresesi,Atik Ali Paşa Medresesi,historic,12,0.687220,member,auto_grouped,Atik Ali Paşa Medresesi
108,499944905,cp_499944905,Atik Ali Paşa Medresesi,Atik Ali Paşa Medresesi,historic,12,0.694578,primary,auto_grouped,Atik Ali Paşa Medresesi


## Save outputs

In [18]:
canonical_pois.to_csv("../data/processed/canonical_pois.csv", index=False)
canonical_poi_members.to_csv("../data/processed/canonical_poi_members.csv", index=False)
safe_merge_pairs_df.to_csv("../data/processed/canonical_safe_merge_pairs.csv", index=False)

print("Saved: ../data/processed/canonical_pois.csv")
print("Saved: ../data/processed/canonical_poi_members.csv")
print("Saved: ../data/processed/canonical_safe_merge_pairs.csv")
print("canonical_pois:", canonical_pois.shape)
print("canonical_poi_members:", canonical_poi_members.shape)

Saved: ../data/processed/canonical_pois.csv
Saved: ../data/processed/canonical_poi_members.csv
Saved: ../data/processed/canonical_safe_merge_pairs.csv
canonical_pois: (146, 17)
canonical_poi_members: (150, 10)
